# iRage AlgoArena — Short-Horizon Return Prediction | v12

## v12 = v11 + 6 timing fixes (all 3 models fully trained)

### v11 Problems
- Feature selection consumed 909s (56% of budget) — 4 compounding bugs
- CatBoost got only 43s remaining — 0 folds completed
- LGB_Shallow cut at fold 3 — partial OOF
- CATBOOST_DONE=True bug always included zero-signal CAT in ensemble

### v12 Fixes
| Fix | Change | Saved |
|---|---|---|
| FS: num_leaves 31→15 | Use main model leaves in selection | ~450s |
| FS: rounds 300→80 | Presence detection saturates at ~50 rounds | ~250s |
| FS: no val_set | Remove per-round eval overhead | ~90s |
| FS: MIN_FOLDS 3→4 | Actually filter to ~150 universal features | +quality |
| LGB guard: 0.9→reserve 450s for CAT | CAT guaranteed budget | CAT works |
| LGB_EARLY_STOP 200→100 | Best_iter unchanged, saves tail waste | ~70s |
| LGB_Shallow early_stop 100→50 | 7-leaf trees plateau fast | ~25s |
| CAT: remove iter_budget formula | Stale CAT_SECS_PER_ITER=0.231 broke iter cap | CAT works |
| CAT: iterations 1200→600 | od_wait=50 stops at ~232 avg anyway | cleaner |
| CAT DONE guard: n_done>=3 | Only include reliable CAT in ensemble | accuracy |
| Fix CATBOOST_DONE=True bug | Was unconditionally overwriting False | accuracy |
| Fix get_best_iteration()→None | Prevent nan in scorecard | clean output |

## Projected v12 budget: ~858s (762s buffer)
Setup 100s | FS 108s | LGB_Huber 310s | LGB_Shallow 140s | CAT 170s | Ensemble 30s

## Evaluation Formula
Final Score = 2 × mean_batch_R2 − full_test_R2
MIN_FOLDS=4 → ~150 regime-universal features → consistent across batches

In [1]:
# CELL 1 - IMPORTS
import os, gc, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

import lightgbm as lgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score

try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
    print('CatBoost OK')
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not available')

SEED = 23
np.random.seed(SEED)
NOTEBOOK_START = time.time()
print(f'LightGBM {lgb.__version__}')
print('Imports OK')

CatBoost OK
LightGBM 4.6.0
Imports OK


In [2]:
# CELL 2 - CONFIG
#
# v12 timing fixes vs v11:
#   LGB_EARLY_STOP: 200 → 100  (saves ~70s tail waste across 5 LGB_Huber folds)
#   CAT_BUDGET_S: NEW — guarantees 450s for CatBoost regardless of LGB timing
#   CAT iterations: 1200 → 600  (od_wait=50 stops at ~232 avg; 1200 was never reached)
#   REMOVED: CAT_SECS_PER_ITER = 0.231  (stale v4 calibration caused broken iter_budget formula)

class CFG:
    if os.path.exists('/kaggle/input'):
        DATA_DIR = Path('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage')
    else:
        DATA_DIR = Path(r'F:\#Project\iRAGE_Hackathon\algo11\kaggle\input\competitions\short-horizon-return-prediction-challenge-by-i-rage')
    TRAIN_PATH = DATA_DIR / 'train.parquet'
    TEST_PATH  = DATA_DIR / 'test.parquet'
    SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'
    if os.path.exists('/kaggle/working'):
        OUTPUT_DIR = Path('/kaggle/working')
    else:
        OUTPUT_DIR = Path('./output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    N_FOLDS = 5
    SEED    = 23

    TARGET_CLIP = True
    CLIP_LOWER  = 2
    CLIP_UPPER  = 98

    BUILD_MOMENTUM     = True
    BUILD_ABS_MOMENTUM = True
    BUILD_FAMILY_AGG   = True
    BUILD_PRICE_FEATS  = True
    BUILD_SO3_POLY     = True

    N_TOP_FEATURES     = 300

    STANDARDIZE        = True

    # ── MODEL A: LGB with Huber loss ──────────────────────────
    LGB_HUBER_PARAMS = {
        'objective':         'huber',
        'alpha':             0.02,
        'metric':            'huber',
        'learning_rate':     0.02,
        'num_leaves':        15,
        'max_depth':         -1,
        'min_child_samples': 200,
        'feature_fraction':  0.6,
        'bagging_fraction':  0.8,
        'bagging_freq':      1,
        'lambda_l1':         0.5,
        'lambda_l2':         8.0,
        'min_gain_to_split': 0.0,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED,
    }

    # ── MODEL B: LGB Shallow (7 leaves = universal patterns only) ─
    LGB_SHALLOW_PARAMS = {
        'objective':         'huber',
        'alpha':             0.02,
        'metric':            'huber',
        'learning_rate':     0.05,
        'num_leaves':        7,
        'max_depth':         -1,
        'min_child_samples': 500,
        'feature_fraction':  0.5,
        'bagging_fraction':  0.8,
        'bagging_freq':      1,
        'lambda_l1':         1.0,
        'lambda_l2':         15.0,
        'min_gain_to_split': 0.0,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED + 2,
    }

    # ── MODEL C: CatBoost with Huber loss ─────────────────────
    # v12 FIX: iterations 1200→600 (od_wait=50 stops at ~232 avg; was never reaching 1200)
    # v12 FIX: REMOVED CAT_SECS_PER_ITER (stale 0.231 from v4 caused broken iter_budget formula)
    CAT_PARAMS = dict(
        loss_function='Huber:delta=0.05',
        eval_metric='RMSE',
        iterations=600,           # was 1200 — od_wait=50 stops at ~232 avg anyway
        learning_rate=0.04,
        depth=5,
        l2_leaf_reg=10.0,
        min_data_in_leaf=150,
        border_count=128,
        random_seed=SEED,
        od_type='Iter',
        od_wait=50,
        verbose=200,
        task_type='CPU',
        thread_count=-1,
        allow_writing_files=False,
    )

    LGB_NUM_ROUNDS = 5000
    LGB_EARLY_STOP = 100       # v12 FIX: was 200 — saves ~70s tail waste (best_iter unchanged)

    RUNTIME_LIMIT_S = 1620     # 27 min hard stop
    CAT_BUDGET_S    = 450      # v12 NEW: guaranteed seconds reserved for CatBoost
    # REMOVED: CAT_SECS_PER_ITER = 0.231  — stale v4 calibration, caused broken iter_budget formula

print('Config OK')


Config OK


In [3]:
# CELL 3 - DATA LOADING
print('Loading data...')
t0 = time.time()

train = pd.read_parquet(CFG.TRAIN_PATH)
test  = pd.read_parquet(CFG.TEST_PATH)

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Loaded in {time.time()-t0:.1f}s')

train_ids = train['ID'].values.copy()
test_ids  = test['ID'].values.copy()
y_raw     = train['TARGET'].copy()

if 'CV_GROUP' in train.columns:
    cv_groups    = train['CV_GROUP'].values.copy()
    USE_GROUP_CV = True
    n_groups     = len(np.unique(cv_groups))
    print(f'CV_GROUP: {n_groups} groups -> GroupKFold')
    train.drop(columns=['ID', 'TARGET', 'CV_GROUP'], inplace=True)
else:
    cv_groups    = None
    USE_GROUP_CV = False
    print('No CV_GROUP -> KFold fallback')
    train.drop(columns=['ID', 'TARGET'], inplace=True)

test.drop(columns=['ID'], inplace=True)

assert list(train.columns) == list(test.columns), 'FATAL: column mismatch!'
print(f'Features: {train.shape[1]} | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Loading data...
Train: (661574, 448) | Test: (410139, 446)
Loaded in 19.0s
CV_GROUP: 70 groups -> GroupKFold
Features: 445 | Elapsed: 35s


In [4]:
# CELL 4 - TARGET CLIPPING
# P2/P98: proven better than P1/P99 (v5 vs OLD v6 comparison).
# Removes extreme microstructure noise (kurtosis=48).
# TRAINING ONLY. Test predictions unconstrained.

desc = y_raw.describe(percentiles=[0.01, 0.02, 0.05, 0.25, 0.5, 0.75, 0.95, 0.98, 0.99])
print(desc)
print(f'Kurtosis: {y_raw.kurtosis():.2f} (fat tails - justifies Huber/MAE loss)')

clip_lo  = y_raw.quantile(CFG.CLIP_LOWER / 100)
clip_hi  = y_raw.quantile(CFG.CLIP_UPPER / 100)
y_series = y_raw.clip(clip_lo, clip_hi)
n_clip   = int(((y_raw < clip_lo) | (y_raw > clip_hi)).sum())
print(f'Clip P{CFG.CLIP_LOWER}/P{CFG.CLIP_UPPER}: [{clip_lo:.6f}, {clip_hi:.6f}]')
print(f'Clipped: {n_clip:,} rows ({100*n_clip/len(y_raw):.2f}%)')
print(f'Std: {y_raw.std():.6f} -> {y_series.std():.6f}')

y = y_series.values.astype(np.float32)
print('Target ready OK')

count    661574.000000
mean         -0.000036
std           0.036660
min          -1.285424
1%           -0.097335
2%           -0.075900
5%           -0.051777
25%          -0.016153
50%           0.000000
75%           0.016073
95%           0.051537
98%           0.075723
99%           0.097421
max           1.346982
Name: TARGET, dtype: float64
Kurtosis: 48.09 (fat tails - justifies Huber/MAE loss)
Clip P2/P98: [-0.075900, 0.075723]
Clipped: 26,464 rows (4.00%)
Std: 0.036660 -> 0.029513
Target ready OK


In [5]:
# CELL 5 - FEATURE CATEGORIZATION
all_cols  = train.columns.tolist()
lag1_cols = [c for c in all_cols if c.endswith('_LagT1')]
lag2_cols = [c for c in all_cols if c.endswith('_LagT2')]
lag3_cols = [c for c in all_cols if c.endswith('_LagT3')]
lag_set   = set(lag1_cols + lag2_cols + lag3_cols)
base_cols = [c for c in all_cols if c not in lag_set]

special_cols   = ['Price', 'SO3_T']
pure_base_cols = [c for c in base_cols if c not in special_cols]

def get_family(col):
    return col.split('_')[0]

families = sorted(set(get_family(c) for c in pure_base_cols))

lag_map = {
    c: {
        'lag1': c+'_LagT1' if c+'_LagT1' in lag_set else None,
        'lag2': c+'_LagT2' if c+'_LagT2' in lag_set else None,
        'lag3': c+'_LagT3' if c+'_LagT3' in lag_set else None,
    } for c in pure_base_cols
}

print(f'Pure base: {len(pure_base_cols)} | Lag1/2/3: {len(lag1_cols)}/{len(lag2_cols)}/{len(lag3_cols)}')
print(f'Families: {families} | Total: {len(all_cols)}')

Pure base: 110 | Lag1/2/3: 111/111/111
Families: ['S01', 'S02', 'S03', 'S04', 'S05'] | Total: 445


In [6]:
# CELL 6 - FEATURE ENGINEERING
# ALL operations are purely row-wise. No cross-sample test operations.
# Same function applied identically to train and test.

def feature_engineering(df, lag_map, families, pure_base_cols, verbose=True):
    df = df.copy()
    new_feats = []

    # A. MOMENTUM: Acc12=LagT1-LagT2, Acc23=LagT2-LagT3, Div13=LagT1-LagT3
    if CFG.BUILD_MOMENTUM:
        n0 = len(new_feats)
        for bc, lags in lag_map.items():
            l1, l2, l3 = lags['lag1'], lags['lag2'], lags['lag3']
            if l1 and l2 and l1 in df.columns and l2 in df.columns:
                df[bc+'_Acc12'] = df[l1]-df[l2]; new_feats.append(bc+'_Acc12')
            if l2 and l3 and l2 in df.columns and l3 in df.columns:
                df[bc+'_Acc23'] = df[l2]-df[l3]; new_feats.append(bc+'_Acc23')
            if l1 and l3 and l1 in df.columns and l3 in df.columns:
                df[bc+'_Div13'] = df[l1]-df[l3]; new_feats.append(bc+'_Div13')
        if verbose: print(f'  Momentum: {len(new_feats)-n0}')

    # B. |LagT1|: momentum magnitude (trees cannot learn |x| from x alone)
    if CFG.BUILD_ABS_MOMENTUM:
        n0 = len(new_feats)
        for bc, lags in lag_map.items():
            l1 = lags['lag1']
            if l1 and l1 in df.columns:
                df[bc+'_AbsMom'] = df[l1].abs(); new_feats.append(bc+'_AbsMom')
        if verbose: print(f'  AbsMom:   {len(new_feats)-n0}')

    # C. Price-specific: Price_Acc, Price_Div13, Price_AbsMom
    if CFG.BUILD_PRICE_FEATS and 'Price' in df.columns:
        n0 = len(new_feats)
        pl1, pl2, pl3 = 'Price_LagT1', 'Price_LagT2', 'Price_LagT3'
        if pl1 in df.columns and pl2 in df.columns:
            df['Price_Acc']    = df[pl1]-df[pl2]; new_feats.append('Price_Acc')
        if pl1 in df.columns and pl3 in df.columns:
            df['Price_Div13']  = df[pl1]-df[pl3]; new_feats.append('Price_Div13')
        if pl2 in df.columns and pl3 in df.columns:
            df['Price_Acc23']  = df[pl2]-df[pl3]; new_feats.append('Price_Acc23')
        if pl1 in df.columns:
            df['Price_AbsMom'] = df[pl1].abs();   new_feats.append('Price_AbsMom')
        if verbose: print(f'  Price:    {len(new_feats)-n0}')

    # D. Family aggregates: cross-instrument consensus / dispersion signals
    if CFG.BUILD_FAMILY_AGG:
        n0 = len(new_feats)
        for fam in families:
            fb = [c for c in pure_base_cols if get_family(c)==fam and c in df.columns]
            fl1 = [c+'_LagT1' for c in fb if c+'_LagT1' in df.columns]
            if len(fb) < 2: continue
            df[fam+'_BaseMean'] = df[fb].mean(axis=1); new_feats.append(fam+'_BaseMean')
            df[fam+'_BaseStd']  = df[fb].std(axis=1);  new_feats.append(fam+'_BaseStd')
            if len(fl1) >= 2:
                fmom = df[fl1]
                df[fam+'_MomMean']   = fmom.mean(axis=1); new_feats.append(fam+'_MomMean')
                df[fam+'_MomStd']    = fmom.std(axis=1);  new_feats.append(fam+'_MomStd')
                df[fam+'_MomSpread'] = fmom.max(axis=1)-fmom.min(axis=1); new_feats.append(fam+'_MomSpread')
        if verbose: print(f'  FamilyAgg:{len(new_feats)-n0}')

    # E. SO3_T polynomial + interactions
    if CFG.BUILD_SO3_POLY and 'SO3_T' in df.columns:
        n0 = len(new_feats)
        df['SO3_T_sq'] = df['SO3_T']**2; new_feats.append('SO3_T_sq')
        df['SO3_T_cb'] = df['SO3_T']**3; new_feats.append('SO3_T_cb')
        if 'Price_LagT1' in df.columns:
            df['SO3T_x_PriceMom'] = df['SO3_T']*df['Price_LagT1']
            new_feats.append('SO3T_x_PriceMom')
        for fam in families:
            fn = fam+'_MomMean'
            if fn in df.columns:
                df['SO3T_x_'+fam+'_Mom'] = df['SO3_T']*df[fn]
                new_feats.append('SO3T_x_'+fam+'_Mom')
        if verbose: print(f'  SO3_T:    {len(new_feats)-n0}')

    if verbose: print(f'  Total new: {len(new_feats)} | Grand total: {df.shape[1]}')
    return df, new_feats


print('Running FE...')
t0 = time.time()
train_fe, new_feats = feature_engineering(train, lag_map, families, pure_base_cols)
test_fe,  _         = feature_engineering(test,  lag_map, families, pure_base_cols, verbose=False)
print(f'Done in {time.time()-t0:.1f}s | Train: {train_fe.shape} | Test: {test_fe.shape}')

Running FE...
  Momentum: 330
  AbsMom:   110
  Price:    4
  FamilyAgg:20
  SO3_T:    7
  Total new: 471 | Grand total: 916
Done in 53.4s | Train: (661574, 916) | Test: (410139, 916)


In [7]:
# CELL 7 - FINALIZE ARRAYS
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace( [np.inf, -np.inf], np.nan, inplace=True)
assert list(train_fe.columns) == list(test_fe.columns), 'FATAL: column mismatch!'

feature_cols = train_fe.columns.tolist()
n_features   = len(feature_cols)
print(f'Total features: {n_features}')

X      = train_fe.values.astype(np.float32)
X_test = test_fe.values.astype(np.float32)
print(f'X: {X.shape} | X_test: {X_test.shape}')

del train_fe, test_fe, train, test
gc.collect()
print(f'Memory freed OK | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Total features: 916
X: (661574, 916) | X_test: (410139, 916)
Memory freed OK | Elapsed: 288s


In [8]:
# CELL 8 - CROSS-FOLD FEATURE SELECTION (v12)
# v11 bugs (consumed 909s = 56% of budget for zero extra feature reduction):
#   Bug 1: num_leaves=31 override → 5.5x cost vs main model (800×31 vs 300×15)
#   Bug 2: num_boost_round=300 → presence saturates at ~50 rounds, 250 rounds wasted
#   Bug 3: valid_sets=[_d_val] with no early stopping → eval overhead every round
#   Bug 4: MIN_FOLDS=3 selected 300 features same as v8 — 909s saved nothing
#
# v12 fixes:
#   Fix 1: Remove 'num_leaves':31 override → uses num_leaves=15 from LGB_HUBER_PARAMS
#   Fix 2: num_boost_round=80 → sufficient for gain>0 presence detection
#   Fix 3: No val_set → removes per-round eval overhead (~15% per fold)
#   Fix 4: MIN_FOLDS=4 → 80% regime presence → genuine ~150 universal features
#
# v12 projected cost: 800×15 leaves × 80 rounds × 5 folds ≈ 108s (was 909s)

print(f'Cross-fold feature selection: {len(feature_cols)} features -> universal subset...')
t_fs = time.time()

if USE_GROUP_CV:
    _spl_fs = list(GroupKFold(n_splits=CFG.N_FOLDS).split(X, y, groups=cv_groups))
else:
    _spl_fs = list(KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(X, y))

# v12 FIX: no 'num_leaves':31 override → uses num_leaves=15 from main model
# Rationale: goal is gain>0 detection only, not precise importance ranking.
# num_leaves=15 finds presence just as reliably as 31, at half the cost per iter.
_fs_params = {
    **CFG.LGB_HUBER_PARAMS,
    'learning_rate': 0.05,   # faster convergence for selection pass
    # REMOVED: 'num_leaves': 31  ← this single line cost ~450s in v11
}

fold_presence = np.zeros(len(feature_cols), dtype=int)
fold_gains    = np.zeros(len(feature_cols), dtype=np.float64)

for fold_idx, (trn_fs, val_fs) in enumerate(_spl_fs):
    t_fold = time.time()
    _d_trn = lgb.Dataset(X[trn_fs], label=y[trn_fs],
                         feature_name=feature_cols, free_raw_data=True)
    # v12 FIX: no val_set → removes per-round eval overhead
    # We only need gain>0 presence, not validation metric tracking

    _m = lgb.train(
        _fs_params, _d_trn,
        num_boost_round=80,    # v12 FIX: was 300 — 80 rounds sufficient for gain>0 detection
        # REMOVED: valid_sets=[_d_val]
        callbacks=[lgb.log_evaluation(-1)],
    )
    fold_imp = _m.feature_importance(importance_type='gain').astype(np.float64)
    fold_presence += (fold_imp > 0).astype(int)
    fold_gains    += fold_imp
    print(f'  Fold {fold_idx+1}/5: {(fold_imp>0).sum()} non-zero features | {time.time()-t_fold:.0f}s')

    del _m, _d_trn
    gc.collect()

# v12 FIX: MIN_FOLDS=4 (was 3)
# With MIN_FOLDS=3, v11 selected 300 features = same as v8 (909s saved nothing).
# MIN_FOLDS=4 requires 80% regime presence → genuine ~150 universal features.
MIN_FOLDS = 4
stable_mask = fold_presence >= MIN_FOLDS
n_stable    = stable_mask.sum()
n_all_folds = (fold_presence == 5).sum()
n_4plus     = (fold_presence >= 4).sum()

print(f'\nFeature presence across 5 folds:')
for k in range(6):
    count = (fold_presence == k).sum()
    print(f'  Present in exactly {k} folds: {count} features')

stable_avg_gain = np.where(stable_mask, fold_gains / CFG.N_FOLDS, -1.0)
sorted_idx   = np.argsort(stable_avg_gain)[::-1]
n_keep       = min(CFG.N_TOP_FEATURES, n_stable)
sel_idx      = sorted_idx[:n_keep]
sel_feats    = [feature_cols[i] for i in sel_idx]

imp_df = pd.DataFrame({
    'feature':   feature_cols,
    'n_folds':   fold_presence,
    'avg_gain':  fold_gains / CFG.N_FOLDS,
    'selected':  [i in sel_idx for i in range(len(feature_cols))],
})
imp_df = imp_df.sort_values(['n_folds','avg_gain'], ascending=False)
imp_df.to_csv(CFG.OUTPUT_DIR / 'feature_importance.csv', index=False)

X          = X[:, sel_idx]
X_test     = X_test[:, sel_idx]
feature_cols = sel_feats
n_features   = len(feature_cols)

print(f'\nSelected {n_features} cross-fold stable features')
print(f'  All-5-folds stable: {n_all_folds}')
print(f'  4+ folds stable:    {n_4plus}')
print(f'  3+ folds stable:    {(fold_presence>=3).sum()}')
print(f'  MIN_FOLDS threshold: {MIN_FOLDS} (kept top {n_keep})')
print(f'Top 5: {sel_feats[:5]}')
print(f'Done in {time.time()-t_fs:.1f}s | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')
del imp_df
gc.collect()


Cross-fold feature selection: 916 features -> universal subset...
  Fold 1/5: 348 non-zero features | 42s
  Fold 2/5: 346 non-zero features | 39s
  Fold 3/5: 343 non-zero features | 38s
  Fold 4/5: 347 non-zero features | 39s
  Fold 5/5: 348 non-zero features | 40s

Feature presence across 5 folds:
  Present in exactly 0 folds: 339 features
  Present in exactly 1 folds: 134 features
  Present in exactly 2 folds: 106 features
  Present in exactly 3 folds: 100 features
  Present in exactly 4 folds: 99 features
  Present in exactly 5 folds: 138 features

Selected 237 cross-fold stable features
  All-5-folds stable: 138
  4+ folds stable:    237
  3+ folds stable:    337
  MIN_FOLDS threshold: 4 (kept top 237)
Top 5: ['S03_A02_D03_W02_LagT1', 'S01_F03_U01_LagT1', 'Price_LagT3', 'Price_LagT1', 'S03_A02_D04_W02_LagT1']
Done in 199.8s | Elapsed: 510s


0

In [9]:
# CELL 9 - Z-SCORE STANDARDIZATION (TRAIN STATS ONLY)
# Removes inter-family scale mismatch. Train stats only -> no test leakage.
# Policy compliant: uses only simple aggregate statistics from training data.

if CFG.STANDARDIZE:
    t_std     = time.time()
    tr_mean   = np.nanmean(X, axis=0).astype(np.float32)
    tr_std    = np.nanstd(X,  axis=0).astype(np.float32)
    tr_std    = np.where(tr_std < 1e-8, 1.0, tr_std)

    X      = np.clip((X      - tr_mean) / tr_std, -10, 10).astype(np.float32)
    X_test = np.clip((X_test - tr_mean) / tr_std, -10, 10).astype(np.float32)
    X      = np.nan_to_num(X,      nan=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0)

    print(f'Standardized | X mean={X.mean():.4f} std={X.std():.4f} | {time.time()-t_std:.1f}s')

print(f'Final arrays: X={X.shape} X_test={X_test.shape} y={y.shape}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Standardized | X mean=-0.0002 std=0.9332 | 4.7s
Final arrays: X=(661574, 237) X_test=(410139, 237) y=(661574,)
Elapsed: 519s


In [10]:
# CELL 10 - CV SPLITTER
if USE_GROUP_CV:
    splitter = GroupKFold(n_splits=CFG.N_FOLDS)
    splits   = list(splitter.split(X, y, groups=cv_groups))
    print(f'GroupKFold on CV_GROUP ({n_groups} groups -> {CFG.N_FOLDS} folds)')
    for i, (ti, vi) in enumerate(splits):
        print(f'  Fold {i+1}: {len(ti):,} train | {len(vi):,} val | {len(np.unique(cv_groups[vi]))} groups')
else:
    splits = list(KFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(X, y))
    print('KFold fallback')
print('CV ready OK')

GroupKFold on CV_GROUP (70 groups -> 5 folds)
  Fold 1: 530,676 train | 130,898 val | 14 groups
  Fold 2: 528,902 train | 132,672 val | 14 groups
  Fold 3: 528,902 train | 132,672 val | 14 groups
  Fold 4: 528,908 train | 132,666 val | 14 groups
  Fold 5: 528,908 train | 132,666 val | 14 groups
CV ready OK


In [11]:
# CELL 11 - GENERIC LGB TRAINER
# v12 FIX: guard changed from 'elapsed > 0.9*limit and fold>=3'
#          to 'elapsed > limit - CAT_BUDGET_S and fold>=2'
# Rationale: old guard fired at 1458s leaving only 162s for CAT (not enough).
# New guard fires when CAT_BUDGET_S (450s) remain — guarantees CAT gets full budget.

def run_lgb_cv(params, num_rounds, early_stop, splits, X, y, X_test,
               feature_cols, name='LGB', runtime_limit=CFG.RUNTIME_LIMIT_S):
    oof   = np.zeros(len(y), dtype=np.float64)
    ptest = np.zeros(len(X_test), dtype=np.float64)
    scores, iters = [], []

    print(f'--- {name} ---')
    for fold, (ti, vi) in enumerate(splits):
        elapsed = time.time() - NOTEBOOK_START
        # v12 FIX: reserve CAT_BUDGET_S for CatBoost, allow guard from fold 2
        # old: elapsed > runtime_limit * 0.9 and fold >= 3  (fired at 1458s)
        # new: elapsed > runtime_limit - CAT_BUDGET_S and fold >= 2  (fires at 1170s)
        cat_budget = getattr(CFG, 'CAT_BUDGET_S', 450)
        if elapsed > (runtime_limit - cat_budget) and fold >= 2:
            print(f'  Time budget ({elapsed:.0f}s): stopping {name} at fold {fold}')
            break

        t0   = time.time()
        dtr  = lgb.Dataset(X[ti], label=y[ti], feature_name=feature_cols, free_raw_data=True)
        dval = lgb.Dataset(X[vi], label=y[vi], reference=dtr,             free_raw_data=True)

        m = lgb.train(
            params, dtr,
            num_boost_round=num_rounds,
            valid_sets=[dtr, dval],
            valid_names=['train','valid'],
            callbacks=[
                lgb.early_stopping(early_stop, verbose=False),
                lgb.log_evaluation(500),
            ],
        )
        bi = m.best_iteration
        iters.append(bi)

        vp        = m.predict(X[vi], num_iteration=bi)
        oof[vi]   = vp
        ptest    += m.predict(X_test, num_iteration=bi) / CFG.N_FOLDS

        fr2 = r2_score(y[vi], vp)
        scores.append(fr2)
        print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fr2:+.6f} | iter: {bi} | {time.time()-t0:.0f}s')

        del m, dtr, dval
        gc.collect()

    n_done = len(scores)
    if n_done < CFG.N_FOLDS:
        ptest = ptest * CFG.N_FOLDS / n_done
        fi    = np.concatenate([splits[i][1] for i in range(n_done)])
        oof_r2 = r2_score(y[fi], oof[fi])
    else:
        oof_r2 = r2_score(y, oof)

    print(f'  {name} OOF R2: {oof_r2:+.6f} | std: {np.std(scores):.6f} | avg_iter: {np.mean(iters):.0f}')
    return oof, ptest, iters, scores, oof_r2

print('LGB trainer defined OK')


LGB trainer defined OK


In [12]:
# CELL 12 - RUN ALL LGB MODELS
# v12 FIX: LGB_Shallow early_stop 100→50
# Rationale: 7-leaf trees plateau fast; 50 rounds patience is generous.
# best_iter unchanged (still stops at same quality point, just 50 rounds later not 100).
# Saves ~25s across folds.

print('=' * 55)
print(f'LightGBM CV - {CFG.N_FOLDS} folds x 2 models')
print('v8: LGB_Q50 removed (0.000784 OOF v7 = weakest, same cost)')
print('=' * 55)

oof_lgb_huber, pred_lgb_huber, iters_huber, scores_huber, r2_lgb_huber = run_lgb_cv(
    CFG.LGB_HUBER_PARAMS, CFG.LGB_NUM_ROUNDS, CFG.LGB_EARLY_STOP,
    splits, X, y, X_test, feature_cols, name='LGB_Huber'
)

oof_lgb_shal, pred_lgb_shal, iters_shal, scores_shal, r2_lgb_shal = run_lgb_cv(
    CFG.LGB_SHALLOW_PARAMS, CFG.LGB_NUM_ROUNDS, 50,  # v12 FIX: was 100 → 50
    splits, X, y, X_test, feature_cols, name='LGB_Shallow'
)

lgb_iters = iters_huber
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')


LightGBM CV - 5 folds x 2 models
v8: LGB_Q50 removed (0.000784 OOF v7 = weakest, same cost)
--- LGB_Huber ---
[500]	train's huber: 0.000283821	valid's huber: 0.000278601
  Fold 1/5 | R2: -0.000271 | iter: 496 | 42s
[500]	train's huber: 0.000283294	valid's huber: 0.000280836
  Fold 2/5 | R2: +0.001044 | iter: 818 | 57s
[500]	train's huber: 0.000280496	valid's huber: 0.000291872
[1000]	train's huber: 0.000278066	valid's huber: 0.000291782
  Fold 3/5 | R2: +0.001381 | iter: 1047 | 68s
[500]	train's huber: 0.000279703	valid's huber: 0.000295231
  Fold 4/5 | R2: +0.000342 | iter: 510 | 42s
[500]	train's huber: 0.000284079	valid's huber: 0.000277476
[1000]	train's huber: 0.000281681	valid's huber: 0.00027734
  Fold 5/5 | R2: +0.003195 | iter: 1105 | 71s
  LGB_Huber OOF R2: +0.001187 | std: 0.001176 | avg_iter: 795
--- LGB_Shallow ---
  Fold 1/5 | R2: -0.000561 | iter: 356 | 22s
[500]	train's huber: 0.000283423	valid's huber: 0.000280914
[1000]	train's huber: 0.000281309	valid's huber: 0.0002

In [13]:
# CELL 13 - CATBOOST WITH HUBER LOSS
# v12 FIXES (4 bugs from v11):
#
# Bug 1: iter_budget formula used stale CAT_SECS_PER_ITER=0.231 (from v4)
#        With 96s remaining: iter_budget=31 → max(400,31)=400 → ran all 400 iter → overran budget
#        Fix: REMOVE iter_budget formula entirely. Use CAT_PARAMS.iterations=600 + od_wait=50.
#
# Bug 2: remaining < 60 threshold too tight for a full CAT fold (~35s/fold)
#        Fix: raise to remaining < 90 for cleaner fold boundary
#
# Bug 3: CATBOOST_DONE = True ran UNCONDITIONALLY after the if/elif/else block
#        This overwrote the False set in n_done==0 branch → broken CAT always in ensemble
#        Fix: set CATBOOST_DONE inside conditional branches only; require n_done>=3
#
# Bug 4: get_best_iteration() returns None when max_iter reached without od_wait firing
#        → cat_iters.append(None) → np.mean([None]) → nan in scorecard
#        Fix: fallback to m.tree_count_ - 1 when get_best_iteration() returns None

oof_cat    = np.zeros(len(y), dtype=np.float64)
pred_cat   = np.zeros(len(X_test), dtype=np.float64)
cat_scores, cat_iters = [], []
CATBOOST_DONE = False

if CATBOOST_AVAILABLE:
    print('=' * 55)
    print(f'CatBoost Huber - {CFG.N_FOLDS} folds (PRIMARY)')
    print('=' * 55)

    for fold, (ti, vi) in enumerate(splits):
        elapsed   = time.time() - NOTEBOOK_START
        remaining = CFG.RUNTIME_LIMIT_S - elapsed
        print(f'  Fold {fold+1} | {remaining:.0f}s remaining')

        # v12 FIX (Fix 6c): add 0.85*limit guard to stop CAT gracefully
        # Doc table Fix #6: 'elapsed > RUNTIME_LIMIT_S * 0.85 and fold >= 2'
        # This is a secondary safety net — Cell 11's CAT_BUDGET_S guard already
        # ensures CAT starts with ~450s remaining, so this rarely fires.
        # But it provides defense-in-depth if a fold takes longer than expected.
        if elapsed > CFG.RUNTIME_LIMIT_S * 0.85 and fold >= 2:
            print(f'  Time budget ({elapsed:.0f}s): stopping CatBoost at fold {fold}')
            break

        # v12 FIX: threshold 60→90 — needs more buffer for a clean CAT fold
        if remaining < 90:
            print('  TIME EXHAUSTED - stopping CatBoost')
            break

        # v12 FIX: REMOVE the entire iter_budget / CAT_SECS_PER_ITER formula.
        # Old code: iters_fold = min(1200, max(400, int((remaining-60)/folds_left/0.231)))
        # Problem: with remaining=96s → iter_budget=31 → max(400,31)=400
        #          best at iter 393, od_wait=50 → runs all 400 → overruns budget.
        # Solution: CAT_PARAMS already has iterations=600 with od_wait=50.
        #           Let early stopping do its job. Average convergence is ~232 iter.
        # p_fold = CFG.CAT_PARAMS  (direct, no iteration override)

        t0 = time.time()
        pool_tr = Pool(X[ti], y[ti])
        pool_va = Pool(X[vi], y[vi])

        # v12 FIX: use CFG.CAT_PARAMS directly (no {**CAT_PARAMS, 'iterations': iters_fold})
        m = CatBoostRegressor(**CFG.CAT_PARAMS)
        m.fit(pool_tr, eval_set=pool_va, use_best_model=True)

        vp          = m.predict(X[vi])
        oof_cat[vi] = vp
        pred_cat   += m.predict(X_test) / CFG.N_FOLDS

        fr2 = r2_score(y[vi], vp)
        # v12 FIX: get_best_iteration() can return None if max_iter reached without od_wait
        bi  = m.get_best_iteration()
        if bi is None:
            bi = m.tree_count_ - 1   # fallback: last iteration index
        cat_scores.append(fr2)
        cat_iters.append(bi)
        print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fr2:+.6f} | best_iter: {bi} | {time.time()-t0:.0f}s')

        del m, pool_tr, pool_va
        gc.collect()

    n_done = len(cat_scores)
    if n_done == 0:
        print('No CatBoost folds completed — skipping safely')
        cat_r2 = 0.0
        pred_cat = np.zeros(len(X_test))
        CATBOOST_DONE = False  # correctly stays False
    elif n_done < CFG.N_FOLDS:
        pred_cat = pred_cat * CFG.N_FOLDS / n_done
        fi = np.concatenate([splits[i][1] for i in range(n_done)])
        cat_r2 = r2_score(y[fi], oof_cat[fi])
        # v12 FIX: only include partial CAT if 3+ folds ran (otherwise too unreliable)
        CATBOOST_DONE = (n_done >= 3)
        if not CATBOOST_DONE:
            print(f'  CAT only {n_done} fold(s) — excluding from ensemble (unreliable)')
    else:
        cat_r2 = r2_score(y, oof_cat)
        CATBOOST_DONE = True

    cat_std = np.std(cat_scores) if cat_scores else 0.0
    # v12 FIX: REMOVED the unconditional 'CATBOOST_DONE = True' that was here in v11
    # That line overwrote the False set in the n_done==0 branch above.

    avg_iter_str = f'{np.mean(cat_iters):.0f}' if cat_iters else 'n/a'
    print(f'CAT OOF R2: {cat_r2:+.6f} | std: {cat_std:.6f} | avg_iter: {avg_iter_str}')
    print(f'CATBOOST_DONE: {CATBOOST_DONE} (n_done={n_done})')
    print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')
else:
    cat_r2 = 0.0
    print('CatBoost not available')


CatBoost Huber - 5 folds (PRIMARY)
  Fold 1 | 677s remaining
0:	learn: 0.0296130	test: 0.0290985	best: 0.0290985 (0)	total: 234ms	remaining: 2m 20s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.02909458601
bestIteration = 12

Shrink model to first 13 iterations.
  Fold 1/5 | R2: +0.000203 | best_iter: 12 | 11s
  Fold 2 | 666s remaining
0:	learn: 0.0295918	test: 0.0291911	best: 0.0291911 (0)	total: 98.9ms	remaining: 59.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.02918192736
bestIteration = 21

Shrink model to first 22 iterations.
  Fold 2/5 | R2: +0.000636 | best_iter: 21 | 10s
  Fold 3 | 656s remaining
0:	learn: 0.0294094	test: 0.0299168	best: 0.0299168 (0)	total: 91ms	remaining: 54.5s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.02990963238
bestIteration = 11

Shrink model to first 12 iterations.
  Fold 3/5 | R2: +0.000540 | best_iter: 11 | 8s
  Fold 4 | 648s remaining
0:	learn: 0.0293475	test: 0.0301584	best: 0.0

In [14]:
# CELL 14 - ENSEMBLE OPTIMIZATION
# OOF-based Nelder-Mead weight optimization.
# NO test-set information used. Softmax weights sum to 1.

model_names = ['LGB_Huber', 'LGB_Shallow']
oofs_list   = [oof_lgb_huber, oof_lgb_shal]
preds_list  = [pred_lgb_huber, pred_lgb_shal]

if CATBOOST_DONE:
    model_names.append('CAT_Huber')
    oofs_list.append(oof_cat)
    preds_list.append(pred_cat)

print('ENSEMBLE OPTIMIZATION')
print('-' * 50)
for nm, oof in zip(model_names, oofs_list):
    print(f'  {nm:<14} OOF R2: {r2_score(y, oof):+.6f}')

# Simple average
oof_sa  = np.mean(oofs_list, axis=0)
pred_sa = np.mean(preds_list, axis=0)
r2_sa   = r2_score(y, oof_sa)
print(f'Simple avg:  {r2_sa:+.6f}')

# Optimized weights
oofs_arr  = np.stack(oofs_list, axis=1)
preds_arr = np.stack(preds_list, axis=1)

def neg_r2(rw):
    w = np.exp(rw) / np.exp(rw).sum()
    return -r2_score(y, (oofs_arr * w).sum(axis=1))

best_r2_opt, best_w = -np.inf, None
for s in range(15):
    rng = np.random.RandomState(s)
    w0  = rng.randn(len(oofs_list)) * 0.1
    res = minimize(neg_r2, w0, method='Nelder-Mead',
                   options={'maxiter': 3000, 'xatol': 1e-7, 'fatol': 1e-9})
    if -res.fun > best_r2_opt:
        best_r2_opt = -res.fun
        best_w = np.exp(res.x) / np.exp(res.x).sum()

oof_opt  = (oofs_arr  * best_w).sum(axis=1)
pred_opt = (preds_arr * best_w).sum(axis=1)
r2_opt   = r2_score(y, oof_opt)

print(f'Optimized:   {r2_opt:+.6f}')
for nm, w in zip(model_names, best_w):
    print(f'  Weight {nm}: {w:.4f}')

if r2_opt > r2_sa:
    final_oof, final_pred = oof_opt, pred_opt
    final_r2, method_str  = r2_opt, 'Optimized wts'
else:
    final_oof, final_pred = oof_sa, pred_sa
    final_r2, method_str  = r2_sa, 'Simple avg'

print(f'FINAL: {final_r2:+.6f} ({method_str})')

ENSEMBLE OPTIMIZATION
--------------------------------------------------
  LGB_Huber      OOF R2: +0.001187
  LGB_Shallow    OOF R2: +0.001028
  CAT_Huber      OOF R2: +0.001028
Simple avg:  +0.001804
Optimized:   +0.001873
  Weight LGB_Huber: 0.4409
  Weight LGB_Shallow: 0.0894
  Weight CAT_Huber: 0.4697
FINAL: +0.001873 (Optimized wts)


In [19]:
# CELL 15 - SUBMISSION GUARDS AND SAVE
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({'ID': test_ids, 'TARGET': final_pred})

assert len(submission)            == len(test_ids),   'FATAL: row count'
assert list(submission.columns)  == ['ID','TARGET'],  'FATAL: columns'
assert submission['ID'].nunique() == len(test_ids),   'FATAL: dup IDs'
assert not submission['TARGET'].isnull().any(),        'FATAL: NaN'
assert not np.isinf(submission['TARGET'].values).any(),'FATAL: Inf'
assert submission['TARGET'].std() > 0,                 'FATAL: constant'

print('All 6 guards passed OK')

sub_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

v = pd.read_csv(sub_path)
assert len(v) == len(test_ids)
assert list(v.columns) == ['ID','TARGET']

total_elapsed = time.time() - NOTEBOOK_START
print(f'submission.csv SAVED | {len(submission):,} rows | {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print(submission['TARGET'].describe())

All 6 guards passed OK
submission.csv SAVED | 410,139 rows | 1793s = 29.9 min
count    410139.000000
mean          0.000093
std           0.001344
min          -0.007104
25%          -0.000680
50%           0.000143
75%           0.000918
max           0.007424
Name: TARGET, dtype: float64


In [15]:
# CELL 16 - FINAL SCORECARD
total_elapsed = time.time() - NOTEBOOK_START

print('=' * 70)
print('  R2 SCORECARD - iRage Short-Horizon Return Prediction v11')
print('=' * 70)
print(f'  Features orig -> selected: {n_features + len(feature_cols) - len(feature_cols)} -> {n_features}')
print(f'  Train: {len(X):,} | Test: {len(X_test):,}')
print(f'  CV: {"GroupKFold(CV_GROUP)" if USE_GROUP_CV else "KFold(shuffle)"}')
print(f'  Runtime: {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print('-' * 70)
print('  MODEL OOF R2')
for nm, oof in zip(model_names, oofs_list):
    r2  = r2_score(y, oof)
    bar = '#' * min(50, max(0, int(r2 * 1000)))
    print(f'  {nm:<14} {r2:+.6f}  {bar}')
print('-' * 70)
print('  ENSEMBLE')
print(f'  Simple avg:    {r2_sa:+.6f}')
print(f'  Optimized:     {r2_opt:+.6f}')
print(f'  FINAL ({method_str}): {final_r2:+.6f}')
print('-' * 70)
print('  OVERFIT DIAGNOSTIC')
print(f'  LGB_Huber avg best_iter: {np.mean(iters_huber):.0f}')
print(f'  LGB_Shallow avg best_iter: {np.mean(iters_shal):.0f}')
if CATBOOST_DONE: print(f'  CAT avg best_iter: {np.mean(cat_iters):.0f}')
print(f'  NOTE: Train R2 >> Val R2 is expected in this problem.')
print(f'  Focus on: Val R2 > 0 and consistent across folds.')
print('-' * 70)
print('  v11 CHANGES vs v8 (single targeted fix)')
print('  PROBLEM: LGB fold1=-0.000415 (regime-specific features selected)')
print('  FIX: Cross-fold feature selection (features in >=3 of 5 folds)')
print('  RATIONALE: a feature important only in fold5 regimes = regime noise')
print('             a feature important in 3+ folds = universal signal')
print('  EXPECTED: fewer features (~100-200 vs 300), smaller fold std')
print('  EVERYTHING ELSE: identical to v8')
print(f'  Features selected: {n_features} (v8 had 300)')
if CATBOOST_DONE:
    print(f'  CAT avg_iter: {np.mean(cat_iters):.0f} (v8 had 182)')
print('-' * 70)
print('  COMPLIANCE')
print('  OK Data: /kaggle/input/competitions/...')
print('  OK No internet / pip / external data')
print('  OK GroupKFold on CV_GROUP')
print('  OK Zero cross-sample test operations')
print('  OK Feature normalization uses train stats only')
runtime_ok = total_elapsed < 1800
print(f'  {"OK" if runtime_ok else "FAIL"} Runtime {total_elapsed/60:.1f} min {"< 30 min" if runtime_ok else "> 30 min"}')
print('=' * 70)
print()

if final_r2 > 0.005:  print('EXCEPTIONAL - top leaderboard')
elif final_r2 > 0.003:print('STRONG - prize range')
elif final_r2 > 0.002:print('GOOD - above majority')
elif final_r2 > 0.001:print('MODERATE - above baseline')
elif final_r2 > 0:    print('WEAK - positive signal')
else:                  print('NO SIGNAL')

print()
print('NOTEBOOK COMPLETE v11')
# Stability check: did cross-fold selection reduce fold variance?
all_fold_r2s = scores_huber + scores_shal
if CATBOOST_DONE: all_fold_r2s += cat_scores
neg_folds = sum(1 for r in all_fold_r2s if r < 0)
print(f'Negative R2 folds: {neg_folds} (v8 had 3 negative LGB folds)')
if neg_folds == 0:
    print('SUCCESS: Cross-fold selection eliminated negative folds!')
elif neg_folds < 3:
    print(f'PARTIAL: Reduced from 3 to {neg_folds} negative folds.')
else:
    print('NOTE: Still some negative folds. Regime difficulty is irreducible.')

  R2 SCORECARD - iRage Short-Horizon Return Prediction v11
  Features orig -> selected: 237 -> 237
  Train: 661,574 | Test: 410,139
  CV: GroupKFold(CV_GROUP)
  Runtime: 1457s = 24.3 min
----------------------------------------------------------------------
  MODEL OOF R2
  LGB_Huber      +0.001187  #
  LGB_Shallow    +0.001028  #
  CAT_Huber      +0.001028  #
----------------------------------------------------------------------
  ENSEMBLE
  Simple avg:    +0.001804
  Optimized:     +0.001873
  FINAL (Optimized wts): +0.001873
----------------------------------------------------------------------
  OVERFIT DIAGNOSTIC
  LGB_Huber avg best_iter: 795
  LGB_Shallow avg best_iter: 568
  CAT avg best_iter: 59
  NOTE: Train R2 >> Val R2 is expected in this problem.
  Focus on: Val R2 > 0 and consistent across folds.
----------------------------------------------------------------------
  v11 CHANGES vs v8 (single targeted fix)
  PROBLEM: LGB fold1=-0.000415 (regime-specific features selected

---
## SUBMISSION CHECKLIST (v7)

- [ ] Runtime < 30 min (Cell 16)
- [ ] LGB_Huber avg best_iter > 50 (fixed params should now run longer)
- [ ] Final OOF R2 > 0.001476 (beat OLD v6)
- [ ] Fold std < 0.003 per model (stable across regimes)
- [ ] submission.csv: 410,139 rows, columns [ID, TARGET]
- [ ] No NaN, no Inf in TARGET

## Requirement Audit

| Rule | Status |
|---|---|
| `/kaggle/input/competitions/...` exact path | PASS |
| No internet / pip install | PASS |
| CPU only (task_type=CPU) | PASS |
| No cross-sample test ranking/KNN/cluster | PASS |
| GroupKFold on CV_GROUP | PASS |
| Feature selection/normalization: train stats only | PASS |
| 30 min runtime | PASS (dynamic guard active) |